<img src="https://cros.ec.europa.eu/system/files/inline-images/logo%20official%20estp_0.jpg" height="120px">

# ADVANCED PYTHON FOR OFFICIAL STATISTICS
## ICON-Institute · Cologne · June 22–26, 2026 · Day 2 PM
### [Dr. Christian Kauth](https://www.linkedin.com/in/ckauth/)

---

# 🕸️ Notebook 04 — Web Scraping & SQLAlchemy
## *Advanced Python for Official Statistics*

> *"If the data is on the web, Python can get it. If the data is in a database, Python can query it."*

This afternoon we connect to two key external data sources: the **web** and **relational databases**.

| Topic | You will be able to… |
|-------|---------------------|
| **BeautifulSoup** | Parse HTML tables from statistical websites |
| **requests** | Fetch web pages with error handling and caching |
| **Ethical scraping** | Respect `robots.txt`, rate-limit, cache responses |
| **SQLAlchemy Core** | Connect to SQLite; write raw SQL with bound parameters |
| **SQLAlchemy ORM** | Map Python classes to database tables |
| **SILC in SQL** | Load Parquet into a normalised schema; compute AROP with SQL |

In [ ]:
import os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = find_project_root(Path().resolve())
os.chdir(PROJECT_ROOT)
DATA_DIR    = PROJECT_ROOT / "data"
PARQUET_DIR = DATA_DIR / "parquet"
print(f"✅ Project root: {PROJECT_ROOT}")


✅ Project root: D:\Local\ICON\Python_Advanced\python_advanced\src\notebooks


### `pyproject.toml` update — Notebook 04

This notebook adds: **requests>=2.31, beautifulsoup4>=4.12, lxml>=5.4, sqlalchemy>=2.0**

Run the cell below to update `pyproject.toml` and install the new packages.

In [ ]:
%%writefile pyproject.toml
# Day 4: adds web scraping and SQLAlchemy
# New in notebook 04:
#   + requests>=2.31
#   + beautifulsoup4>=4.12
#   + lxml>=5.4
#   + sqlalchemy>=2.0

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=0.20",
    "requests>=2.31",
    "beautifulsoup4>=4.12",
    "lxml>=5.4",
    "sqlalchemy>=2.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
ignore = ["E501"]

[tool.ruff.lint.isort]
known-first-party = ["silc_toolkit"]


Overwriting pyproject.toml


In [ ]:
# Sync all dependencies declared in pyproject.toml
# Run this after every %%writefile pyproject.toml cell
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 04 deps installed")
else:
    print("uv error:", result.stderr[-500:])

silc-toolkit + notebook 04 deps installed


---
## 1 · Web Scraping with BeautifulSoup

### Ethical Scraping Rules

1. Check `robots.txt` — respect disallowed paths
2. Rate-limit (sleep 1–2 s between requests)
3. Cache responses locally — don't re-download what you have
4. Use a descriptive `User-Agent`
5. Prefer official APIs (Eurostat has `eurostat` Python package)

In [ ]:
import time
from pathlib import Path
import requests
from bs4 import BeautifulSoup

cache_dir = PROJECT_ROOT / ".scrape_cache"
cache_dir.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "ICON-Institute SILC course / educational"
}


def fetch_cached(url: str, cache_path: Path, delay: float = 1.0) -> str:
    """Fetch URL once; return cached content on subsequent calls."""
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")
    time.sleep(delay)
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    cache_path.write_text(response.text, encoding="utf-8")
    return response.text


SILC_URL   = "https://ec.europa.eu/eurostat/web/microdata/public-microdata/statistics-on-income-and-living-conditions"
cache_file = cache_dir / "eurostat_silc_overview.html"

try:
    html = fetch_cached(SILC_URL, cache_file)
    soup = BeautifulSoup(html, "lxml")
    print(f"✅ Page fetched: {len(html):,} bytes")
    print(f"   Title: {soup.title.string if soup.title else 'N/A'}")
except Exception as e:
    print(f"⚠️  Could not fetch Eurostat page: {e}")
    print("   Working offline — using local catalogue metadata instead")
    html, soup = None, None

✅ Page fetched: 219,821 bytes
   Title: Statistics on income and living conditions - Microdata - Eurostat


### 1.1 · Rule #1 in Practice — Check `robots.txt`

Before scraping at scale, ask the server what it permits. Python's standard-library
`urllib.robotparser` reads a site's `robots.txt` and tells you whether your user-agent
is allowed to fetch a given path — no third-party packages needed.


In [ ]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urlsplit

def may_fetch(url: str, user_agent: str = HEADERS["User-Agent"]) -> bool:
    """Rule #1 in action: honour the site's robots.txt before scraping."""
    parts = urlsplit(url)
    robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"
    print(robots_url)
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
    except Exception as exc:
        print(f"⚠️  Could not read {robots_url} ({exc}) — proceeding cautiously")
        return True
    return rp.can_fetch(user_agent, url)

allowed = may_fetch(SILC_URL)
print(f"robots.txt allows scraping the SILC microdata page? {allowed}")


https://ec.europa.eu/robots.txt
robots.txt allows scraping the SILC microdata page? True


### 1.2 · Scraping at Scale — Every Country's Download URL

In **notebook 01** we *hard-coded* a `DOWNLOAD_URLS` dictionary mapping each country to its
PUF `.zip` file. Those URLs were copied by hand from this very page — tedious and brittle:
the moment Eurostat re-publishes the files, every link might change.

Let's do it properly. The download links are ordinary `<a href="…_PUF_EUSILC.zip…">` anchors,
so one `find_all` pass plus a regex rebuilds the **entire** dictionary automatically —
the link **text** gives the country name, the **href** gives the URL.


In [ ]:
import re
import pandas as pd

ZIP_RE = re.compile(r"/([A-Z]{2})_PUF_EUSILC\.zip", re.IGNORECASE)


def extract_download_urls(soup) -> dict[str, dict]:
    """Return {country_code: {'name': link text, 'url': href}} for every PUF zip link."""
    found: dict[str, dict] = {}
    if soup is None:
        return found
    for a in soup.find_all("a", href=True):
        match = ZIP_RE.search(a["href"])
        if match:
            code = match.group(1).upper()
            found[code] = {"name": a.get_text(strip=True) or code, "url": a["href"]}
    return found


scraped = extract_download_urls(soup)
print(f"✅ Scraped {len(scraped)} country download URLs from the live page")

# The canonical mapping other code (e.g. the notebook-01 download loop) consumes
DOWNLOAD_URLS = {code: rec["url"] for code, rec in sorted(scraped.items())}

df_urls = pd.DataFrame(
    [{"code": c, "country": r["name"], "url": r["url"]} for c, r in sorted(scraped.items())]
)
print(f"   {len(df_urls)} countries — first rows:")
df_urls.head(8)


✅ Scraped 25 country download URLs from the live page
   25 countries — first rows:


,code,country,url
0,AT,Austria,https://ec.europa.eu/eurostat/documents/203647...
1,BE,Belgium,https://ec.europa.eu/eurostat/documents/203647...
2,BG,Bulgaria,https://ec.europa.eu/eurostat/documents/203647...
3,CY,Cyprus,https://ec.europa.eu/eurostat/documents/203647...
4,DE,Germany,https://ec.europa.eu/eurostat/documents/203647...
5,DK,Denmark,https://ec.europa.eu/eurostat/documents/203647...
6,EE,Estonia,https://ec.europa.eu/eurostat/documents/203647...
7,EL,Greece,https://ec.europa.eu/eurostat/documents/203647...


In [ ]:
print(df_urls.to_string(index=False))

code        country                                                                                                                            url
  AT        Austria https://ec.europa.eu/eurostat/documents/203647/16979414/AT_PUF_EUSILC.zip/96eaa4a3-762a-66e2-7a71-7459d329a762?t=1686929857578
  BE        Belgium https://ec.europa.eu/eurostat/documents/203647/16979414/BE_PUF_EUSILC.zip/7368be7b-6554-3de5-54e6-65e45d9d5399?t=1686929853723
  BG       Bulgaria https://ec.europa.eu/eurostat/documents/203647/16979414/BG_PUF_EUSILC.zip/cd0807cf-b3e6-c13f-ba6c-a3e34bfaab14?t=1686929858419
  CY         Cyprus https://ec.europa.eu/eurostat/documents/203647/16979414/CY_PUF_EUSILC.zip/690f30d6-ccb5-4821-1436-27a2574fc3f5?t=1686929859188
  DE        Germany https://ec.europa.eu/eurostat/documents/203647/16979414/DE_PUF_EUSILC.zip/239569fc-96da-4fb3-892f-221233e18c94?t=1686929867541
  DK        Denmark https://ec.europa.eu/eurostat/documents/203647/16979414/DK_PUF_EUSILC.zip/7de0a3c4-6e98-95fc-9db8-

### 1.3 · Companion Resources on the Same Page

The download URLs aren't the only thing worth scraping. The microdata landing page
also links to the **variable metadata workbook**, the **datasets-availability table**,
and the **annual guidelines**. A robust scraper classifies *every* link so you can
build a complete catalogue of a dataset's companion resources in one pass.


In [ ]:
# The same page links to companion resources — not just the data itself.
# Classify every link by what kind of resource it points to.
RESOURCE_PATTERNS = {
    "PUF zip":            re.compile(r"_PUF_EUSILC\.zip", re.IGNORECASE),
    "metadata (xlsm)":    re.compile(r"metadata", re.IGNORECASE),
    "availability table": re.compile(r"availab", re.IGNORECASE),
    "guidelines":         re.compile(r"guidelines|circabc", re.IGNORECASE),
}

resources = []
if soup is not None:
    for a in soup.find_all("a", href=True):
        href, label_text = a["href"], a.get_text(strip=True)
        for kind, pat in RESOURCE_PATTERNS.items():
            if pat.search(href) or pat.search(label_text):
                resources.append({"kind": kind, "text": label_text[:45], "url": href})
                break

if resources:
    df_res = pd.DataFrame(resources).drop_duplicates("url").reset_index(drop=True)
    print(f"✅ Classified {len(df_res)} links on the page by resource type:\n")
    print(df_res["kind"].value_counts().to_string())
    print("\nNon-data companion resources (metadata, availability, guidelines):")
    print(df_res[df_res["kind"] != "PUF zip"][["kind", "text"]].to_string(index=False))
else:
    print("⚠️  Offline — companion-resource scraping needs the live page")


✅ Classified 30 links on the page by resource type:

kind
PUF zip               25
metadata (xlsm)        2
guidelines             2
availability table     1

Non-data companion resources (metadata, availability, guidelines):
              kind                           text
   metadata (xlsm)                       Metadata
        guidelines         Manuals and guidelines
        guidelines              annual guidelines
availability table    datasets availability table
   metadata (xlsm) SILC public use files metadata


### 1.4 HTML Table to Pandas

In [ ]:
from IPython.display import HTML, display

example_html = '''<table>
  <thead><tr>
    <th>Country</th><th>AROP (%)</th><th>Gini</th><th>S80/S20</th>
  </tr></thead>
  <tbody>
    <tr><td>BE</td><td>15.3</td><td>26.5</td><td>3.9</td></tr>
    <tr><td>EL</td><td>23.1</td><td>34.3</td><td>6.6</td></tr>
    <tr><td>ES</td><td>22.2</td><td>34.2</td><td>6.5</td></tr>
    <tr><td>FR</td><td>14.1</td><td>30.5</td><td>4.8</td></tr>
    <tr><td>HU</td><td>14.0</td><td>26.9</td><td>4.0</td></tr>
    <tr><td>IE</td><td>16.5</td><td>30.9</td><td>5.0</td></tr>
    <tr><td>LU</td><td>13.4</td><td>28.0</td><td>4.3</td></tr>
    <tr><td>SE</td><td>14.1</td><td>24.8</td><td>3.7</td></tr>
  </tbody>
</table>'''

display(HTML(example_html))

Country,AROP (%),Gini,S80/S20
BE,15.3,26.5,3.9
EL,23.1,34.3,6.6
ES,22.2,34.2,6.5
FR,14.1,30.5,4.8
HU,14.0,26.9,4.0
IE,16.5,30.9,5.0
LU,13.4,28.0,4.3
SE,14.1,24.8,3.7


In [ ]:
soup_ex = BeautifulSoup(example_html, "lxml")
table   = soup_ex.find("table")
headers = [th.get_text(strip=True) for th in table.find_all("th")]
rows    = [
    [td.get_text(strip=True) for td in tr.find_all("td")]
    for tr in table.find("tbody").find_all("tr")
]

df_pub = pd.DataFrame(rows, columns=headers)
for col in ["AROP (%)", "Gini", "S80/S20"]:
    df_pub[col] = df_pub[col].astype(float)

print("Parsed HTML table (simulated Eurostat published indicators 2012):")
df_pub

Parsed HTML table (simulated Eurostat published indicators 2012):


,Country,AROP (%),Gini,S80/S20
0,BE,15.3,26.5,3.9
1,EL,23.1,34.3,6.6
2,ES,22.2,34.2,6.5
3,FR,14.1,30.5,4.8
4,HU,14.0,26.9,4.0
5,IE,16.5,30.9,5.0
6,LU,13.4,28.0,4.3
7,SE,14.1,24.8,3.7


### 1.5 · Browser Automation — When JavaScript Gets in the Way

`requests` + `BeautifulSoup` work wonderfully for **static HTML** — the server sends the
full page and you parse it. Many modern sites, however, render their content with
**JavaScript after the initial load**: the raw HTML you receive is nearly empty, and the
data only appears once a browser has executed the JS. Classic scraping breaks entirely.

The solution is **browser automation**: drive a real (headless) browser from Python so that
JS executes, dynamic content loads, and you can then read the rendered DOM — or interact
with the page just like a human would (click buttons, fill forms, navigate menus).

Two libraries dominate the Python ecosystem:

| Library | Install | Strengths |
|---------|---------|-----------|
| **Selenium** [`selenium · PyPI`](https://pypi.org/project/selenium/) | `pip install selenium` | Long-established, broad browser support, large community |
| **Playwright** [`playwright · PyPI`](https://pypi.org/project/playwright/) | `pip install playwright` then `playwright install` | Modern async API, faster, built-in waiting, all major browsers |

Both let you launch Chrome/Firefox/Edge programmatically, navigate to a URL, wait for
elements to appear, extract text, click links — essentially anything a human does in a browser.

---

#### 🎮 Fun Aside — Playwright MCP + Claude Desktop = AI that browses the web

Playwright also ships as an **MCP server** (`@playwright/mcp`), which exposes browser
actions as tools that an AI agent can call. Connect it to **Claude Desktop** and the model
can navigate, read, and interact with any website autonomously.

For example: point Claude at the [Eurostat quiz](https://ec.europa.eu/eurostat/cache/quiz/)
and add a second MCP server giving access to the
[Eurostat database](https://ec.europa.eu/eurostat/data/database) — Claude then *reads the
question*, *queries the official statistics*, and *clicks the correct answer*, all by itself.
The video below shows exactly that in action (see cell below ▼).

To wire it up, add these two entries to `claude_desktop_config.json`
(**Claude Desktop → Customize → Developer**):

```json
{
  "mcpServers": {
    "playwright": {
      "command": "npx",
      "args": ["@playwright/mcp@latest"]
    }
  }
}
```


In [ ]:
from IPython.display import HTML, display

video_id = "OVrh-WzZpu4"
display(HTML(f'''
<a href="https://youtube.com/shorts/{video_id}" target="_blank">
  <img src="https://img.youtube.com/vi/{video_id}/0.jpg"
       style="height:560px; cursor:pointer;"
       title="Click to open on YouTube"/>
</a>
'''))

---
### 🏋️ Exercise 1 — Read & Debug the Scraper

You don't have to write this scraper from scratch — it's already in the cell below, and it
works. The real skill is being able to **understand exactly what each line does**. So put the
debugging tools from **Notebook 02** to work.

The cell below fetches the Wikipedia *"Member states of the European Union"* page and turns a
messy HTML table into a clean, analysis-ready DataFrame (27 countries × 13 columns). A lot
happens along the way: picking the right table, stripping footnote markers, coercing text to
numbers, and extracting the accession year.

**Your task — step through it, don't rewrite it**

1. **Run the cell** once and look at the output.
2. **Copy the code into a script**, e.g. `eu_members_scraper.py`, wrapping the body in a
   `def scrape_eu_members() -> pd.DataFrame:` with an `if __name__ == "__main__":` block that
   prints the result (remember the imports: `io`, `re`, `pandas`, `requests`, `BeautifulSoup`).
3. **Set breakpoints** (click the gutter) and launch with **`F5` → Debug Python File**
   (Notebook 02 · §7.2). Good lines to pause on:
   - `df_try = pd.read_html(...)` — inspect `df_try.columns` to see the raw, footnoted headers
   - `df_eu["Year"] = ...` — watch `"1 January 1995"` and `"Founder"` become integer years
   - `cleaned = ...` inside the numeric loop — for `src = "Population"`, watch
     `"10,749,635[9]"` lose its footnote before becoming a real number
4. Use **Step Into (`F11`)** on `strip_footnotes(...)` to drop into the helper, then
   **Step Out (`Shift+F11`)** to return to the caller.
5. Read the **Variables** panel after each step and answer:
   - Which of the page's `wikitable`s gets selected, and *why* does the loop stop there?
   - What would break if you stripped footnotes **after** `pd.to_numeric` instead of before?

> 💡 Prefer to stay in the notebook? Click into the cell and use **Run by Line (`F10`)**
> (Notebook 02 · §7.3) to step through it without ever leaving VS Code.


In [ ]:
# 🏋️ Exercise 1 — Study this scraper, then copy it into a .py file and step through it
# (self-contained: defines its own URL + cache path)
import io

WIKI_URL   = "https://en.wikipedia.org/wiki/Member_states_of_the_European_Union"
cache_wiki = cache_dir / "wiki_eu_members.html"


def strip_footnotes(text) -> str:
    """Remove Wikipedia footnote markers like [3] or [4][5] from any string."""
    return re.sub(r"\[.*?\]", "", str(text)).strip()


try:
    html_wiki = fetch_cached(WIKI_URL, cache_wiki)
    soup_wiki = BeautifulSoup(html_wiki, "lxml")
    tables    = soup_wiki.find_all("table", class_="wikitable")

    # Pick the first wikitable that is the full members table (has Country + Accession).
    # NB: pandas 2.x requires a file-like object — wrap the HTML in io.StringIO.
    df_eu = None
    for t in tables:
        df_try     = pd.read_html(io.StringIO(str(t)))[0]
        cols_lower = [str(c).lower() for c in df_try.columns]
        if any("country" in c for c in cols_lower) and any("accession" in c for c in cols_lower):
            df_eu = df_try
            break

    if df_eu is None:
        raise ValueError("EU member-states table not found on the page")

    # 1️⃣ Clean the column headers (drop footnote markers like [3], [4][5])
    df_eu.columns = [strip_footnotes(c) for c in df_eu.columns]

    # 2️⃣ Clean the free-text columns
    for col in ["Country", "Currency", "Largest city", "Official languages"]:
        if col in df_eu.columns:
            df_eu[col] = df_eu[col].map(strip_footnotes)

    # 3️⃣ Turn the messy Accession text into an integer year ("Founder" → 1958)
    df_eu["Year"] = (
        df_eu["Accession"].str.extract(r"(\d{4})")[0].fillna("1958").astype(int)
    )

    # 4️⃣ Coerce the numeric columns to real numbers, then give them tidy names.
    #    Strip footnotes FIRST — otherwise "10,749,635[9]" would merge the "9".
    numeric_cols = {
        "Population":          "Population",
        "Area (km2)":          "Area_km2",
        "GDP (US$ M)":         "GDP_USD_M",
        "GDP (PPP) per cap.":  "GDP_PPP_per_capita",
        "Gini":                "Gini",
        "HDI":                 "HDI",
        "MEPs":                "MEPs",
    }
    for src in numeric_cols:
        if src in df_eu.columns:
            cleaned = (
                df_eu[src].map(strip_footnotes)
                .str.replace(r"[,\s]", "", regex=True)
            )
            df_eu[src] = pd.to_numeric(cleaned, errors="coerce")
    df_eu = df_eu.rename(columns=numeric_cols)

    # 5️⃣ Order the columns sensibly and drop the raw Accession text
    ordered = [
        "Country", "ISO", "Year", "Population", "Area_km2", "Largest city",
        "GDP_USD_M", "GDP_PPP_per_capita", "Currency", "Gini", "HDI", "MEPs",
        "Official languages",
    ]
    df_eu = df_eu[[c for c in ordered if c in df_eu.columns]]

    print(f"✅ Parsed {len(df_eu)} EU member states × {df_eu.shape[1]} columns from Wikipedia:")
    print(f"   Columns: {', '.join(df_eu.columns)}")
except Exception as e:
    print(f"Could not fetch ({e}) — using embedded fallback")
    eu_members = [
        ("Belgium", 1958, "BE"), ("France", 1958, "FR"),
        ("Germany", 1958, "DE"), ("Ireland", 1973, "IE"),
        ("Greece", 1981, "EL"), ("Spain", 1986, "ES"),
        ("Hungary", 2004, "HU"), ("Sweden", 1995, "SE"),
    ]
    df_eu = pd.DataFrame(eu_members, columns=["Country", "Year", "Code"])
    print(df_eu.to_string(index=False))
df_eu


✅ Parsed 27 EU member states × 13 columns from Wikipedia:
   Columns: Country, ISO, Year, Population, Area_km2, Largest city, GDP_USD_M, GDP_PPP_per_capita, Currency, Gini, HDI, MEPs, Official languages


,Country,ISO,Year,Population,Area_km2,Largest city,GDP_USD_M,GDP_PPP_per_capita,Currency,Gini,HDI,MEPs,Official languages
0,Austria,AT,1995,8926000,83855.0,Vienna,535804,73050,euro,29.1,0.930,20,German
1,Belgium,BE,1958,11566041,30528.0,Brussels,662183,73221,euro,33.0,0.951,22,Dutch French German
2,Bulgaria,BG,2007,6916548,110994.0,Sofia,66250,39185,euro,29.2,0.845,17,Bulgarian
3,Croatia,HR,2013,4036355,56594.0,Zagreb,89665,48811,euro,29.0,0.878,12,Croatian
4,Cyprus,CY,2004,896000,9251.0,Nicosia,23380,59858,euro,31.2,0.913,6,Greek Turkish
5,Czech Republic,CZ,2004,10574153,78866.0,Prague,246953,56686,koruna,25.8,0.915,21,Czech
6,Denmark,DK,1973,5833883,43075.0,Copenhagen,347176,83454,krone,24.7,0.962,15,Danish
7,Estonia,EE,2004,1330068,45227.0,Tallinn,43044,48008,euro,36.0,0.905,7,Estonian
8,Finland,FI,1995,5527493,338424.0,Helsinki,306083,64657,euro,26.9,0.948,15,Finnish Swedish
9,France,FR,1958,67439614,632786.0,Paris,2707074,65940,euro,32.7,0.920,81,French


---
## 2 · SQLAlchemy — Python Meets Databases

### The SILC Database Schema

```
countries           households
─────────────       ──────────────────────
country_code PK     household_id  PK
country_name        country       FK→countries
                    survey_year
                    disposable_income   (HY020)
                    equiv_income        (derived)
                    household_size      (HX040)
                    weight              (DB090)
```

In [ ]:
from sqlalchemy import (
    create_engine, text, Integer, Float, String, ForeignKey
)
from sqlalchemy.orm import (
    DeclarativeBase, Session, relationship, mapped_column, Mapped
)

DB_PATH = PROJECT_ROOT / "silc_database.db"
engine  = create_engine(f"sqlite:///{DB_PATH}", echo=False)
print(f"✅ SQLAlchemy engine: {engine.url}")

✅ SQLAlchemy engine: sqlite:///D:\Local\ICON\Python_Advanced\python_advanced\silc_database.db


### 2.1 · ORM Models — `mapped_column`, `relationship`, `back_populates`

SQLAlchemy 2.x uses **typed declarative mapping**: every column and relationship is declared as a Python class attribute with a type annotation.

| Construct | What it does |
|-----------|-------------|
| `mapped_column(...)` | Maps a class attribute to a database column. Arguments control the SQL type (`String`, `Integer`, `Float`), constraints (`primary_key=True`, `nullable=True`), and defaults. The `Mapped[T]` annotation tells SQLAlchemy (and type-checkers) the Python type of the attribute. |
| `relationship(...)` | Declares a **Python-level link** between two ORM classes. It doesn't add a column — the actual join is done via the `ForeignKey` on the child side. SQLAlchemy uses it to load related objects automatically (lazy or eager). |
| `back_populates="..."` | Keeps **both sides of the relationship in sync**. `Country.households` and `Household.country_ref` each name the other; when you append a `Household` to `country.households`, SQLAlchemy automatically sets `household.country_ref` to that `Country` — no extra assignment needed. |

```
Country                         Household
────────────────────            ──────────────────────────────────
country_code  PK  ◄──── FK ─── country
country_name                    survey_year
households  ←─────────────────► country_ref   (relationship pair)
```

`Base.metadata.create_all(engine)` translates these class definitions into `CREATE TABLE` SQL and executes it — one call to materialise the entire schema.

In [ ]:
class Base(DeclarativeBase):
    pass


class Country(Base):
    __tablename__ = "countries"
    country_code: Mapped[str]               = mapped_column(String(2), primary_key=True)
    country_name: Mapped[str | None]        = mapped_column(String(100), nullable=True)
    households:   Mapped[list["Household"]] = relationship(back_populates="country_ref")

    def __repr__(self):
        return f"Country({self.country_code!r}, {self.country_name!r})"


class Household(Base):
    __tablename__ = "households"
    id:                Mapped[int]           = mapped_column(Integer, primary_key=True, autoincrement=True)
    household_id:      Mapped[int]           = mapped_column(Integer)
    country:           Mapped[str]           = mapped_column(String(2), ForeignKey("countries.country_code"))
    survey_year:       Mapped[int]           = mapped_column(Integer)
    disposable_income: Mapped[float]         = mapped_column(Float, default=0.0)
    total_income:      Mapped[float]         = mapped_column(Float, default=0.0)
    household_size:    Mapped[int]           = mapped_column(Integer, default=1)
    weight:            Mapped[float]         = mapped_column(Float, default=1.0)
    equiv_income:      Mapped[float | None]  = mapped_column(Float, nullable=True)
    nuts_region:       Mapped[str | None]    = mapped_column(String(5), nullable=True)
    country_ref:       Mapped["Country"]     = relationship(back_populates="households")

    def __repr__(self):
        return f"Household(id={self.household_id}, {self.country!r} {self.survey_year})"


Base.metadata.create_all(engine)
print("✅ Schema created. Tables:", list(Base.metadata.tables.keys()))

✅ Schema created. Tables: ['countries', 'households']


### 2.2 · Loading Data — Parquet → pandas → SQLite

The loading pipeline has two steps:

1. **Parquet → pandas** — `pd.read_parquet()` reads the pre-built household files from notebook 03, selecting only the columns we need and renaming them to match our schema.
2. **pandas → SQLite** — `DataFrame.to_sql()` bulk-inserts all rows into the `households` table using multi-row `INSERT` batches (`method="multi"`, `chunksize=1000`).

`Session` is used for the `countries` seed data (ORM-style `session.add`), while `to_sql` writes households directly via the engine — a common pattern when inserting large DataFrames where the ORM overhead would be unnecessary.

In [ ]:
# Ensure clean slate (idempotent — safe to re-run)
DB_PATH = PROJECT_ROOT / 'silc_database.db'
engine.dispose()  # close all connections before file delete
if DB_PATH.exists():
    DB_PATH.unlink()
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)
Base.metadata.create_all(engine)

COUNTRY_NAMES = {
    "AT": "Austria",   "BE": "Belgium",   "BG": "Bulgaria",  "CY": "Cyprus",
    "CZ": "Czech Rep", "DE": "Germany",   "DK": "Denmark",   "EE": "Estonia",
    "EL": "Greece",    "ES": "Spain",     "FI": "Finland",   "FR": "France",
    "HR": "Croatia",   "HU": "Hungary",   "IE": "Ireland",   "IT": "Italy",
    "LT": "Lithuania", "LU": "Luxembourg","LV": "Latvia",    "MT": "Malta",
    "NL": "Netherlands","PL": "Poland",   "PT": "Portugal",  "RO": "Romania",
    "SE": "Sweden",    "SI": "Slovenia",  "SK": "Slovakia",
}

PARTICIPANT_COUNTRIES = ["BE", "ES", "FR", "EL", "HU", "IE", "LU", "SE"]

with Session(engine) as session:
    for code, name in COUNTRY_NAMES.items():
        if not session.get(Country, code):
            session.add(Country(country_code=code, country_name=name))
    session.commit()

    for country in PARTICIPANT_COUNTRIES:
        pq_path = PARQUET_DIR / f"{country}_households.parquet"
        if not pq_path.exists():
            print(f"  ⚠️  No Parquet for {country} — run notebook 03 first")
            continue
        df = pd.read_parquet(
            pq_path,
            columns=["HB030", "HB020", "survey_year", "HY020", "HY010",
                     "HX040", "equiv_income"],
        ).rename(columns={
            "HB030": "household_id",
            "HB020": "country",
            "HY020": "disposable_income",
            "HY010": "total_income",
            "HX040": "household_size",
        })
        df["household_size"]    = df["household_size"].fillna(1).clip(lower=1).astype(int)
        df["disposable_income"] = df["disposable_income"].fillna(0.0)
        df["total_income"]      = df["total_income"].fillna(0.0)
        df["equiv_income"]      = df["equiv_income"].fillna(0.0)
        df["weight"]            = 1.0
        df["country"]           = country

        df[["household_id", "country", "survey_year",
            "disposable_income", "total_income", "household_size",
            "weight", "equiv_income"]].to_sql(
                "households",
                engine,
                if_exists="replace", # fail, replace, append
                index=False,
                method="multi",
                chunksize=1000
        )
        print(f"  ✅ {country}: {len(df):,} households loaded")

print("\n✅ Database loading complete")

  ✅ BE: 36,453 households loaded
  ✅ ES: 77,933 households loaded
  ✅ FR: 66,558 households loaded
  ✅ EL: 39,639 households loaded
  ✅ HU: 61,762 households loaded
  ✅ IE: 13,864 households loaded
  ✅ LU: 28,169 households loaded
  ✅ SE: 41,715 households loaded

✅ Database loading complete


### 2.3 · Querying with Raw SQL — `session.execute(text(...))`

SQLAlchemy's `text()` lets you write plain SQL strings while still using the managed connection pool and parameterised binding. The pattern is:

```python
with Session(engine) as session:
    rows = session.execute(text("SELECT ... FROM ... WHERE ...")).fetchall()
```

- **`Session`** opens a connection, manages the transaction, and closes everything on exit.
- **`text(...)`** wraps a literal SQL string. Always use bound parameters (`:param`) for user-supplied values — never f-strings — to prevent SQL injection.
- **`.fetchall()`** returns a list of `Row` objects that behave like named tuples; pass them to `pd.DataFrame(rows, columns=[...])` to get a tidy table.

The two queries below use standard `GROUP BY` aggregates. Note that SQLite **has no `MEDIAN()`** function, so Query 2 approximates it with `AVG()` — a known limitation discussed after the output.

In [ ]:
# Query 1: household counts and mean income by country
with Session(engine) as session:
    rows = session.execute(text(
        "SELECT country, survey_year, COUNT(*) as n_hh, "
        "ROUND(AVG(disposable_income),0) as avg_inc "
        "FROM households WHERE survey_year=2012 "
        "GROUP BY country ORDER BY country"
    )).fetchall()

df_q1 = pd.DataFrame(rows, columns=["Country", "Year", "N_HH", "Avg_Income"])
print("Households and mean income by country, 2012:")
print(df_q1.to_string(index=False))

Households and mean income by country, 2012:
Country  Year  N_HH  Avg_Income
     SE  2012  6628     40136.0


In [ ]:
# Query 2: approximate AROP (using mean as median proxy — SQLite has no MEDIAN())
AROP_SQL = (
    "WITH medians AS ("
    "  SELECT country, survey_year, AVG(equiv_income) AS approx_median"
    "  FROM households WHERE equiv_income > 0 GROUP BY country, survey_year"
    "), "
    "thresholds AS ("
    "  SELECT country, survey_year, 0.60 * approx_median AS threshold FROM medians"
    ") "
    "SELECT h.country, h.survey_year, COUNT(*) AS n_total, "
    "  SUM(CASE WHEN h.equiv_income < t.threshold AND h.equiv_income > 0 "
    "      THEN 1 ELSE 0 END) AS n_at_risk, "
    "  ROUND(100.0 * SUM(CASE WHEN h.equiv_income < t.threshold AND h.equiv_income > 0 "
    "      THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN h.equiv_income > 0 "
    "      THEN 1 ELSE 0 END), 0), 1) AS arop_pct "
    "FROM households h "
    "JOIN thresholds t ON h.country=t.country AND h.survey_year=t.survey_year "
    "WHERE h.survey_year=2012 GROUP BY h.country ORDER BY arop_pct DESC"
)

with Session(engine) as session:
    rows = session.execute(text(AROP_SQL)).fetchall()

df_arop = pd.DataFrame(rows, columns=["Country", "Year", "N_total", "N_at_risk", "AROP_%"])
print("AROP by country 2012 (SQL — mean as median proxy):")
print(df_arop.to_string(index=False))
print()
print("Note: SQLite lacks MEDIAN(). PostgreSQL has:")
print("  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY equiv_income)")

AROP by country 2012 (SQL — mean as median proxy):
Country  Year  N_total  N_at_risk  AROP_%
     SE  2012     6628       1952    30.9

Note: SQLite lacks MEDIAN(). PostgreSQL has:
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY equiv_income)


### 2.4 · Querying with SQLAlchemy Core — `select()`, `.join()`, `.where()`

Instead of writing SQL strings, SQLAlchemy's **expression language** lets you build queries as Python objects. This gives you:

- **Type safety** — column names are attributes on the model class (`Household.country`), so typos are caught by the type checker, not at runtime.
- **Composability** — you can build a base `select(...)` and layer `.where()`, `.order_by()`, etc. on top conditionally.
- **Database portability** — the same Python code generates correct SQL for SQLite, PostgreSQL, MySQL, etc.

| SQL clause | SQLAlchemy equivalent |
|------------|----------------------|
| `SELECT col, AGG(col)` | `select(Model.col, func.agg(Model.col))` |
| `FROM a JOIN b ON ...` | `.join(B, A.col == B.col)` |
| `WHERE col = val` | `.where(Model.col == val)` |
| `GROUP BY col` | `.group_by(Model.col)` |
| `ORDER BY expr DESC` | `.order_by(expr.desc())` |

The result is still executed via `session.execute(stmt).fetchall()` — the session handles the connection just as with raw SQL.

In [ ]:
from sqlalchemy import select, func as sqlfunc

with Session(engine) as session:
    stmt = (
        select(
            Household.country,
            Country.country_name,
            sqlfunc.count(Household.household_id).label("n_hh"),
            sqlfunc.round(sqlfunc.avg(Household.disposable_income), 0).label("mean_income"),
        )
        .join(Country, Household.country == Country.country_code)
        .where(Household.survey_year == 2012)
        .group_by(Household.country)
        .order_by(sqlfunc.avg(Household.disposable_income).desc())
    )
    rows = session.execute(stmt).fetchall()

df_orm = pd.DataFrame(rows, columns=["Code", "Country", "N_HH", "Mean_Income"])
print("ORM query — Mean disposable income 2012 (descending):")
print(df_orm.to_string(index=False))

ORM query — Mean disposable income 2012 (descending):
Code Country  N_HH  Mean_Income
  SE  Sweden  6628      40136.0


In [ ]:
%%writefile silc_toolkit/database.py
"""
SQLAlchemy ORM models and helpers for the SILC SQLite database.
"""
from __future__ import annotations

from pathlib import Path
from typing import List, Optional

from sqlalchemy import Float, ForeignKey, Integer, String, create_engine, text
from sqlalchemy.orm import (
    DeclarativeBase,
    Mapped,
    Session,
    mapped_column,
    relationship,
)


class Base(DeclarativeBase):
    pass


class Country(Base):
    __tablename__ = "countries"

    country_code: Mapped[str] = mapped_column(String(2), primary_key=True)
    country_name: Mapped[Optional[str]] = mapped_column(String(100), nullable=True)
    households: Mapped[List["Household"]] = relationship(back_populates="country_ref")


class Household(Base):
    __tablename__ = "households"

    household_id: Mapped[int] = mapped_column(
        Integer, primary_key=True, autoincrement=False
    )
    country: Mapped[str] = mapped_column(
        String(2), ForeignKey("countries.country_code")
    )
    survey_year: Mapped[int] = mapped_column(Integer)
    disposable_income: Mapped[float] = mapped_column(Float, default=0.0)
    total_income: Mapped[float] = mapped_column(Float, default=0.0)
    household_size: Mapped[int] = mapped_column(Integer, default=1)
    weight: Mapped[float] = mapped_column(Float, default=1.0)
    equiv_income: Mapped[Optional[float]] = mapped_column(Float, nullable=True)
    nuts_region: Mapped[Optional[str]] = mapped_column(String(5), nullable=True)
    country_ref: Mapped["Country"] = relationship(back_populates="households")


def get_engine(db_path: Path):
    return create_engine(f"sqlite:///{db_path}", echo=False)


def create_schema(engine) -> None:
    Base.metadata.create_all(engine)

Overwriting silc_toolkit/database.py


---
### 🏋️ Exercise 2 — Gini in SQLite via Custom Aggregate

SQLite doesn't have `GINI()` built-in, but Python lets you register custom aggregates.

```python
import sqlite3

class GiniAggregate:
    def __init__(self): self._values = []
    def step(self, v):
        if v and v > 0: self._values.append(v)
    def finalize(self):
        from silc_toolkit.indicators import gini_coefficient
        return gini_coefficient(self._values)

conn.create_aggregate("GINI", 1, GiniAggregate)
```

Register it via `engine.raw_connection()` and write a query computing `GINI(equiv_income)` per country for 2012.

In [ ]:
# 🏋️ Exercise 2 — Your solution here!
from silc_toolkit.indicators import gini_coefficient

class GiniAggregate:
    def __init__(self):
        self._values = []

    def step(self, value):
        # TODO: append positive values
        pass

    def finalize(self):
        # TODO: return gini_coefficient(self._values)
        return None

# raw_conn = engine.raw_connection()
# raw_conn.create_aggregate("GINI", 1, GiniAggregate)


<details><summary>💡 Click for the solution</summary>

```python
from silc_toolkit.indicators import gini_coefficient

class GiniAggregate:
    def __init__(self): self._values = []
    def step(self, v):
        if v is not None and v > 0: self._values.append(v)
    def finalize(self): return gini_coefficient(self._values)

raw_conn = engine.raw_connection()
try:
    raw_conn.create_aggregate("GINI", 1, GiniAggregate)
    cur = raw_conn.cursor()
    cur.execute(
        "SELECT country, survey_year, ROUND(GINI(equiv_income), 4) AS gini "
        "FROM households WHERE survey_year=2012 AND equiv_income>0 "
        "GROUP BY country ORDER BY gini DESC"
    )
    rows = cur.fetchall()
finally:
    raw_conn.close()

df_gini = pd.DataFrame(rows, columns=["Country", "Year", "Gini"])
print(df_gini.to_string(index=False))
```
</details>

---
## 🗺️ Recap

| Concept | Key syntax | SILC application |
|---------|-----------|------------------|
| **fetch + cache** | `requests.get()` + `Path.write_text()` | Download metadata once |
| **BeautifulSoup** | `soup.find("table")` / `find_all("tr")` | Parse HTML catalogues |
| **pd.read_html** | `pd.read_html(str(table))[0]` | One-liner HTML tables |
| **SQLAlchemy engine** | `create_engine("sqlite:///path.db")` | Portable DB connection |
| **ORM models** | `class Household(Base): ...` | Tables as Python classes |
| **Session query** | `session.execute(select(...))` | Type-safe query building |
| **Raw SQL** | `session.execute(text(sql), params)` | Complex CTE queries |
| **to_sql** | `df.to_sql("table", engine, if_exists="append")` | Bulk Parquet→SQLite |
| **Custom aggregate** | `conn.create_aggregate("GINI", 1, Class)` | Extend SQLite with Python |

---
## ⏭️ Up Next

**Notebook 05 — NumPy, statsmodels & ML Pipelines** (tomorrow morning)

- Vectorise Gini with NumPy — 100× faster
- Fit OLS regression: income ~ education + age (statsmodels)
- Build a scikit-learn poverty-risk classifier with SHAP explanations

🌙 *Great work today — see you at 9:00 tomorrow!*